In [1]:
!pip install -q laya detoxify
import sys, torch
print(sys.version)
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
# If CUDA shows False after install, restart the session and re-run from the next cell.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 3.4 MB/s eta 0:00:00
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch 2.10.0+cu128 | CUDA: True


In [3]:
import os, time, numpy as np, pandas as pd
BASE = '/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge'
print(os.listdir(BASE))
LABELS = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']

def load(name):
    for ext in ['.csv', '.csv.zip']:
        p = f'{BASE}/{name}{ext}'
        if os.path.exists(p):
            return pd.read_csv(p)
    raise FileNotFoundError(name)

train = load('train')
test = load('test').merge(load('test_labels'), on='id')
test = test[test['toxic'] != -1].reset_index(drop=True)   # drop unscored rows
train = train.drop_duplicates('comment_text').reset_index(drop=True)

print('train', train.shape, '| test (scored)', test.shape)
print(pd.DataFrame({'train_pos': train[LABELS].sum(), 'train_%': (train[LABELS].mean()*100).round(2),
                    'test_pos': test[LABELS].sum()}))

['train.csv.zip', 'sample_submission.csv.zip', 'test_labels.csv.zip', 'test.csv.zip']
train (159571, 8) | test (scored) (63978, 8)
               train_pos  train_%  test_pos
toxic              15294     9.58      6090
severe_toxic        1595     1.00       367
obscene             8449     5.29      3691
threat               478     0.30       211
insult              7877     4.94      3427
identity_hate       1405     0.88       712


In [4]:
from sklearn.model_selection import train_test_split
tr, va = train_test_split(train, test_size=0.1, random_state=42, stratify=train['toxic'])
sample = test.sample(10000, random_state=42)   # same sample for every model (Laya zero-shot is slower)

tr.to_csv('/kaggle/working/train_split.csv', index=False)
va.to_csv('/kaggle/working/val_split.csv', index=False)
test.to_csv('/kaggle/working/test_clean.csv', index=False)
sample.to_csv('/kaggle/working/test_sample10k.csv', index=False)
print(len(tr), len(va), len(test), len(sample))
print('sample positives:', sample[LABELS].sum().to_dict())  # threat will be tiny -> noisy AUC, note it

143613 15958 63978 10000
sample positives: {'toxic': 981, 'severe_toxic': 58, 'obscene': 603, 'threat': 34, 'insult': 580, 'identity_hate': 125}


In [10]:
from sklearn.metrics import roc_auc_score
def score(y, p, name):
    s = {l: roc_auc_score(y[l], p[l]) for l in LABELS}
    s['mean'] = float(np.mean(list(s.values())))
    return pd.Series(s, name=name).round(4)

results, speed = [], []

## Baseline A: TF-IDF + logistic regression

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack

wv = TfidfVectorizer(sublinear_tf=True, strip_accents='unicode', analyzer='word',
                     ngram_range=(1,2), max_features=50000, min_df=2)
cv = TfidfVectorizer(sublinear_tf=True, strip_accents='unicode', analyzer='char_wb',
                     ngram_range=(2,5), max_features=50000, min_df=2)
Xtr = hstack([wv.fit_transform(tr.comment_text), cv.fit_transform(tr.comment_text)]).tocsr()

t0 = time.time()
Xte = hstack([wv.transform(test.comment_text), cv.transform(test.comment_text)]).tocsr()
tfidf_pred = pd.DataFrame(index=test.index)
models = {}
for l in LABELS:
    models[l] = LogisticRegression(C=4, solver='liblinear', max_iter=1000).fit(Xtr, tr[l])
for l in LABELS:
    tfidf_pred[l] = models[l].predict_proba(Xte)[:, 1]
# timing covers vectorizing + predicting the whole test set
t = time.time() - t0
speed.append({'model': 'TF-IDF + LR (CPU)', 'comments_per_sec': round(len(test)/t, 1)})

results.append(score(test, tfidf_pred, 'TF-IDF full test'))
results.append(score(sample, tfidf_pred.loc[sample.index], 'TF-IDF 10k sample'))
pd.DataFrame(results)

,toxic,severe_toxic,obscene,threat,insult,identity_hate,mean
TF-IDF full test,0.9643,0.9831,0.9770,0.992,0.9703,0.985,0.9786
TF-IDF 10k sample,0.9625,0.9795,0.9767,0.992,0.9678,0.983,0.9769


## Baseline B: Detoxify `original`

In [12]:
from detoxify import Detoxify
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
dtx = Detoxify('original', device=dev)
print(dtx.predict(['hello world']).keys())   # check key names match MAP below

MAP = {'toxic':'toxicity','severe_toxic':'severe_toxicity','obscene':'obscene',
       'threat':'threat','insult':'insult','identity_hate':'identity_attack'}

def detox_predict(texts, bs=64):
    out = {k: [] for k in MAP}
    for i in range(0, len(texts), bs):
        r = dtx.predict(texts[i:i+bs])
        for k, v in MAP.items():
            out[k].extend(r[v])
    return pd.DataFrame(out)

t0 = time.time()
detox_pred = detox_predict(test.comment_text.tolist())
t = time.time() - t0
detox_pred.index = test.index
speed.append({'model': f'Detoxify original ({dev})', 'comments_per_sec': round(len(test)/t, 1)})

results.append(score(test, detox_pred, 'Detoxify full test'))
results.append(score(sample, detox_pred.loc[sample.index], 'Detoxify 10k sample'))
pd.DataFrame(results)

Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.1-alpha/toxic_original-c1212f89.ckpt" to /root/.cache/torch/hub/checkpoints/toxic_original-c1212f89.ckpt


100%|██████████| 418M/418M [00:02<00:00, 160MB/s]  


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

dict_keys(['toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack'])


,toxic,severe_toxic,obscene,threat,insult,identity_hate,mean
TF-IDF full test,0.9643,0.9831,0.9770,0.9920,0.9703,0.9850,0.9786
TF-IDF 10k sample,0.9625,0.9795,0.9767,0.9920,0.9678,0.9830,0.9769
Detoxify full test,0.9739,0.9915,0.9824,0.9965,0.9805,0.9928,0.9863
Detoxify 10k sample,0.9736,0.9892,0.9818,0.9950,0.9807,0.9923,0.9854


## Baseline C: Laya zero-shot

In [13]:
from laya import Router
router = Router()

QUESTIONS = {
  'toxic':         {'type':'noul','instructions':'Is this comment rude, disrespectful or unreasonable?'},
  'severe_toxic':  {'type':'noul','instructions':'Is this comment extremely hateful, aggressive or disrespectful?'},
  'obscene':       {'type':'noul','instructions':'Does this comment contain obscene or vulgar language?'},
  'threat':        {'type':'noul','instructions':'Does this comment threaten to harm someone?'},
  'insult':        {'type':'noul','instructions':'Does this comment insult a person?'},
  'identity_hate': {'type':'noul','instructions':'Does this comment attack someone for their identity (race, religion, gender, sexuality, etc.)?'},
}
MAX_CHARS = 1200   # ~300 tokens; Laya English reads ~320 tokens of state

r = router.predict({'comment': test.comment_text.iloc[0][:MAX_CHARS]}, QUESTIONS)
print(type(r)); print(r)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/laya/router.py:260: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  agent = Agent(repo, device=self.device, token=self.token, subfolder=sub)


<class 'dict'>
{'model': 'laya-rl-agent', 'answers': {'toxic': {'type': 'noul', 'noul': 0.053, 'confidence': 0.947, 'answer_confidence': 0.947, 'action': {'act_probability': 1.0}}, 'severe_toxic': {'type': 'noul', 'noul': 0.0367, 'confidence': 0.9633, 'answer_confidence': 0.9633, 'action': {'act_probability': 1.0}}, 'obscene': {'type': 'noul', 'noul': 0.0, 'confidence': 1.0, 'answer_confidence': 1.0, 'action': {'act_probability': 1.0}}, 'threat': {'type': 'noul', 'noul': 0.0, 'confidence': 1.0, 'answer_confidence': 1.0, 'action': {'act_probability': 1.0}}, 'insult': {'type': 'noul', 'noul': 0.0425, 'confidence': 0.9575, 'answer_confidence': 0.9575, 'action': {'act_probability': 1.0}}, 'identity_hate': {'type': 'noul', 'noul': 0.0268, 'confidence': 0.9732, 'answer_confidence': 0.9732, 'action': {'act_probability': 1.0}}}, 'usage': {'input_tokens': 361, 'output_tokens': 0}, 'routing': {'model': 'english', 'repo': 'convaiinnovations/laya', 'reason': 'English Latin text', 'detection': {'sc

In [15]:
def answers_of(r):
    if isinstance(r, dict) and 'answers' in r: return r['answers']
    if hasattr(r, 'answers'): return r.answers
    return r

def to_prob(a):
    if isinstance(a, (int, float)): return float(a)
    if isinstance(a, dict):
        for k in ('noul', 'probability', 'prob', 'p', 'value'):
            if isinstance(a.get(k), (int, float)): return float(a[k])
    raise ValueError(f'Unknown answer shape: {a!r}')

print({k: to_prob(v) for k, v in answers_of(r).items()})   # should print 6 numbers

{'toxic': 0.053, 'severe_toxic': 0.0367, 'obscene': 0.0, 'threat': 0.0, 'insult': 0.0425, 'identity_hate': 0.0268}


In [16]:
from tqdm.auto import tqdm
rows, fails = [], 0
t0 = time.time()
for text in tqdm(sample.comment_text.tolist()):
    try:
        a = answers_of(router.predict({'comment': text[:MAX_CHARS]}, QUESTIONS))
        rows.append({l: to_prob(a[l]) for l in LABELS})
    except Exception as e:
        fails += 1
        rows.append({l: np.nan for l in LABELS})
t = time.time() - t0
laya_zs = pd.DataFrame(rows, index=sample.index)
print('failures:', fails)
speed.append({'model': f'Laya zero-shot, 6 q/comment ({dev})', 'comments_per_sec': round(len(sample)/t, 1)})

ok = laya_zs.dropna().index
results.append(score(sample.loc[ok], laya_zs.loc[ok], 'Laya zero-shot 10k sample'))
laya_zs.to_csv('/kaggle/working/laya_zeroshot_preds.csv')

  0%|          | 0/10000 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

failures: 0


In [18]:
import laya, torch, time

# Load the English model directly (skips the Router's language check)
agent = laya.load("convaiinnovations/laya", device="cuda")
states = [{'comment': t[:MAX_CHARS]} for t in sample.comment_text.tolist()]

# Warm-up so first-call setup isn't counted
agent.predict_batch(states[:32], QUESTIONS, batch_size=16)

# 1) Find the fastest batch size on 1,000 comments
best_bs, best_rate = None, 0
for bs in [8, 16, 32, 64]:
    try:
        torch.cuda.empty_cache()
        t0 = time.time()
        agent.predict_batch(states[:1000], QUESTIONS, batch_size=bs, sort_by_length=True)
        rate = 1000 / (time.time() - t0)
        print(f'batch_size={bs}: {rate:.1f} comments/sec')
        if rate > best_rate: best_bs, best_rate = bs, rate
    except torch.cuda.OutOfMemoryError:
        print(f'batch_size={bs}: out of GPU memory'); break

# 2) Full 10k run with the best batch size
t0 = time.time()
res = agent.predict_batch(states, QUESTIONS, batch_size=best_bs, sort_by_length=True)
t = time.time() - t0
laya_b = pd.DataFrame([{l: to_prob(answers_of(r)[l]) for l in LABELS} for r in res], index=sample.index)

print(f'\nFull 10k: {len(states)/t:.1f} comments/sec (batch_size={best_bs}) vs 11.8 before')
speed.append({'model': f'Laya zero-shot batched bs={best_bs} (cuda)', 'comments_per_sec': round(len(states)/t, 1)})
print(score(sample, laya_b, 'Laya zero-shot batched'))  # should be ~0.971, same as before

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/laya/agent.py:928: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


batch_size=8: 14.9 comments/sec
batch_size=16: 14.9 comments/sec
batch_size=32: 14.7 comments/sec


KeyboardInterrupt: 

## Scoreboard

In [17]:
board = pd.DataFrame(results)
board.to_csv('/kaggle/working/scoreboard.csv')
pd.DataFrame(speed).to_csv('/kaggle/working/speed.csv', index=False)
print(board.to_string()); print(); print(pd.DataFrame(speed).to_string(index=False))

                            toxic  severe_toxic  obscene  threat  insult  identity_hate    mean
TF-IDF full test           0.9643        0.9831   0.9770  0.9920  0.9703         0.9850  0.9786
TF-IDF 10k sample          0.9625        0.9795   0.9767  0.9920  0.9678         0.9830  0.9769
Detoxify full test         0.9739        0.9915   0.9824  0.9965  0.9805         0.9928  0.9863
Detoxify 10k sample        0.9736        0.9892   0.9818  0.9950  0.9807         0.9923  0.9854
Laya zero-shot 10k sample  0.9569        0.9760   0.9546  0.9950  0.9718         0.9704  0.9708

                             model  comments_per_sec
                 TF-IDF + LR (CPU)             209.8
          Detoxify original (cuda)              32.4
Laya zero-shot, 6 q/comment (cuda)              13.3
